# CheXSpark — Stage-1: Lung Segmentation (U-Net)

Huấn luyện **U-Net** phân đoạn trường phổi trên 2,000 ảnh NIH ChestX-ray14 (dev subset),
ground truth là mask **CheXmask** (HybridGNet sinh sẵn).

**Đính kèm 2 dataset vào notebook này (Add data):**
1. Ảnh 224×224: `khanfashee/nih-chest-x-ray-14-224x224-resized`
2. Nhãn + mask RLE: `<tai-khoan>/chexspark-stage2-nih-dev` (dùng chung với Stage-2)

**Vai trò trong hệ thống:** mask do U-Net sinh ra thay cho CheXmask ở thời điểm suy luận
→ Stage-2 (DenseNet121) và Stage-3 (Grad-CAM) không còn phụ thuộc dữ liệu mask bên ngoài.

**Pipeline:** ảnh xám 224² → U-Net (1.9M tham số) → sigmoid → mask phổi 224²
→ loss `BCE + Dice` → eval Dice/IoU.

Chọn **GPU** trước khi chạy (Accelerator → GPU T4 x2 / P100).

In [ ]:
# ============ CONFIG — chỉnh ở đây nếu path Kaggle khác ============
import os

IMG_ROOT  = '/kaggle/input/nih-chest-x-ray-14-224x224-resized'  # ảnh 224x224 PNG
META_ROOT = '/kaggle/input/chexspark-stage2-nih-dev'             # stage2_train.csv
TRAIN_CSV = os.path.join(META_ROOT, 'stage2_train.csv')

IMG_SIZE    = 224
BASE        = 32        # số kênh tầng đầu (local CPU dùng 16; GPU dùng 32)
BATCH_SIZE  = 16
EPOCHS      = 30
LR          = 1e-3
NUM_WORKERS = 2
SEED        = 42

In [ ]:
import os, random, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

In [ ]:
# Index {basename: full_path} để join theo image_index (không đoán cấu trúc folder)
print('Quét ảnh trong', IMG_ROOT, '...')
img_index = {}
for dirpath, _, filenames in os.walk(IMG_ROOT):
    for fn in filenames:
        if fn.lower().endswith('.png'):
            img_index[fn] = os.path.join(dirpath, fn)
print(f'Tìm thấy {len(img_index)} PNG')

In [ ]:
df = pd.read_csv(TRAIN_CSV, dtype={'image_index': str})
df = df[df['mask_left_rle'].notna()].reset_index(drop=True)
df['path'] = df['image_index'].map(img_index)
found = df['path'].notna().sum()
print(f'Rows: {len(df)} | có ảnh trên Kaggle: {found} ({found/len(df):.1%})')
df = df.dropna(subset=['path']).reset_index(drop=True)

tr_df = df[df.split == 'train'].reset_index(drop=True)
va_df = df[df.split == 'val'].reset_index(drop=True)
te_df = df[df.split == 'test'].reset_index(drop=True)
print(f'train/val/test = {len(tr_df)}/{len(va_df)}/{len(te_df)}')

In [ ]:
def decode_rle(rle, h, w):
    '''CheXmask RLE: chuoi cap (start, length), reshape row-major (C order).'''
    vals = [int(v) for v in str(rle).split()]
    flat = np.zeros(h * w, dtype=np.uint8)
    for i in range(0, len(vals), 2):
        flat[vals[i]:vals[i] + vals[i + 1]] = 1
    return flat.reshape((h, w))


class LungSegDataset(Dataset):
    '''(ảnh xám 224², mask phổi 224²). Mask decode 1 lần rồi cache trong RAM.'''

    def __init__(self, frame, img_size=224, train=False):
        self.df = frame.reset_index(drop=True)
        self.img_size = img_size
        self.train = train
        self._masks = {}

    def __len__(self):
        return len(self.df)

    def _mask(self, idx):
        if idx in self._masks:
            return self._masks[idx]
        row = self.df.iloc[idx]
        h, w = int(row['mask_height']), int(row['mask_width'])
        m = decode_rle(row['mask_left_rle'], h, w) | decode_rle(row['mask_right_rle'], h, w)
        m = Image.fromarray((m * 255).astype(np.uint8)).resize(
            (self.img_size, self.img_size), Image.NEAREST)
        arr = (np.array(m) > 127).astype(np.float32)
        self._masks[idx] = arr
        return arr

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('L').resize((self.img_size, self.img_size))
        gray = np.array(img).astype(np.float32) / 255.0
        mask = self._mask(idx)
        if self.train and np.random.rand() < 0.5:
            gray = gray[:, ::-1].copy(); mask = mask[:, ::-1].copy()
        x = torch.from_numpy(((gray - 0.5) / 0.5)[None].copy())
        return x.float(), torch.from_numpy(mask)[None]

tr_ds = LungSegDataset(tr_df, IMG_SIZE, train=True)
va_ds = LungSegDataset(va_df, IMG_SIZE)
te_ds = LungSegDataset(te_df, IMG_SIZE)
tr_dl = DataLoader(tr_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
va_dl = DataLoader(va_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
te_dl = DataLoader(te_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print('DataLoaders ready.')

In [ ]:
# Kiểm tra nhanh: ảnh | mask GT | ảnh đã nhân mask
row = te_df.iloc[0]
img = np.array(Image.open(row['path']).convert('L').resize((IMG_SIZE, IMG_SIZE)))
gt = LungSegDataset(te_df, IMG_SIZE)._mask(0)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title(row['image_index']); ax[0].axis('off')
ax[1].imshow(gt, cmap='gray');   ax[1].set_title('mask phoi (CheXmask)'); ax[1].axis('off')
ax[2].imshow(img * gt, cmap='gray'); ax[2].set_title('masked'); ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    '''U-Net 4 tầng xuống; base=32 → ~7.8M tham số.'''
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1, self.enc2 = ConvBlock(in_ch, b), ConvBlock(b, b * 2)
        self.enc3, self.enc4 = ConvBlock(b * 2, b * 4), ConvBlock(b * 4, b * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(b * 8, b * 16)
        self.up4 = nn.ConvTranspose2d(b * 16, b * 8, 2, stride=2); self.dec4 = ConvBlock(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, 2, stride=2);  self.dec3 = ConvBlock(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, 2, stride=2);  self.dec2 = ConvBlock(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, 2, stride=2);      self.dec1 = ConvBlock(b * 2, b)
        self.out = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


class BCEDiceLoss(nn.Module):
    '''BCE ổn định gradient + Dice chịu mất cân bằng nền/phổi (Dice tính từng ảnh).'''
    def __init__(self, bce_weight=0.5):
        super().__init__(); self.bce_weight = bce_weight
    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)
        p = torch.sigmoid(logits); dims = (1, 2, 3)
        inter = (p * target).sum(dims)
        dice = (2 * inter + 1.0) / (p.sum(dims) + target.sum(dims) + 1.0)
        return self.bce_weight * bce + (1 - self.bce_weight) * (1 - dice.mean())

model = UNet(base=BASE).to(device)
print('params:', sum(p.numel() for p in model.parameters()))
criterion = BCEDiceLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

In [ ]:
def dice_iou(pred, gt, eps=1e-6):
    pred = pred.astype(np.float32); gt = gt.astype(np.float32)
    inter = float((pred * gt).sum())
    return ((2 * inter + eps) / (pred.sum() + gt.sum() + eps),
            (inter + eps) / (pred.sum() + gt.sum() - inter + eps))


@torch.no_grad()
def evaluate(dl):
    model.eval(); dices, ious = [], []
    for x, y in dl:
        prob = torch.sigmoid(model(x.to(device))).float().cpu().numpy()[:, 0]
        for p, g in zip(prob, y.numpy()[:, 0]):
            d, i = dice_iou((p > 0.5).astype(np.float32), g)
            dices.append(d); ious.append(i)
    d, i = np.array(dices), np.array(ious)
    return {'dice_mean': float(d.mean()), 'dice_median': float(np.median(d)),
            'dice_p10': float(np.percentile(d, 10)), 'iou_mean': float(i.mean()),
            'frac_dice_ge_07': float((d >= 0.7).mean())}
print('Loss + eval ready.')

In [ ]:
best, history = -1.0, []
for ep in range(1, EPOCHS + 1):
    model.train(); t0, tot, seen = time.time(), 0.0, 0
    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tot += loss.item() * len(x); seen += len(x)
    scheduler.step()
    va = evaluate(va_dl); tr_loss = tot / len(tr_ds)
    history.append({'epoch': ep, 'train_loss': tr_loss, **{f'val_{k}': v for k, v in va.items()}})
    print(f"Ep{ep:02d}/{EPOCHS} loss={tr_loss:.4f} val_dice={va['dice_mean']:.4f} "
          f"iou={va['iou_mean']:.4f} ({time.time()-t0:.0f}s)")
    if va['dice_mean'] > best:
        best = va['dice_mean']
        torch.save(model.state_dict(), 'best_stage1_unet.pth')
        print(f'  -> checkpoint moi (val_dice={best:.4f})')
print('Best val Dice:', round(best, 4))

In [ ]:
model.load_state_dict(torch.load('best_stage1_unet.pth'))
test = evaluate(te_dl)
print('TEST:', json.dumps(test, indent=2))

In [ ]:
# Lưu model + metrics (Kernel Output) để đăng ký vào MLflow Model Registry
torch.save(model.state_dict(), 'stage1_unet_final.pth')
with open('stage1_metrics.json', 'w') as f:
    json.dump({'best_val_dice': best, 'test': test, 'history': history,
               'config': {'arch': 'unet', 'base': BASE, 'img_size': IMG_SIZE,
                          'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
                          'loss': 'BCEDiceLoss',
                          'ground_truth': 'CheXmask (left|right, khong gom heart)',
                          'n_train': len(tr_df), 'n_val': len(va_df), 'n_test': len(te_df)}},
              f, indent=2)
print('Da luu stage1_unet_final.pth + stage1_metrics.json.')